# Code an LLM Tokenizer from scratch
---

## Step 1 - Creating Tokens

In [162]:
#load data
with open("../data/book-war-and-peace.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total numb er of characters: ", len(raw_text))
print(raw_text[:99])

Total numb er of characters:  3202303
CHAPTER I

"Well, Prince, so Genoa and Lucca are now just family estates of the
Buonapartes. But I 


We will use the regular expression (re) library to split the test.

In [163]:
import re

text = "Hello, how are you? I am fine!"
result = re.split(r'([, ? !]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'how', ' ', 'are', ' ', 'you', '?', '', ' ', 'I', ' ', 'am', ' ', 'fine', '!', '']


The result includes whitespace characters so we need to remove them.

In [164]:
result = [item for item in result if item.strip()] #item.strip() will return false for whitespaces and hence they will not be included
print(result)

['Hello', ',', 'how', 'are', 'you', '?', 'I', 'am', 'fine', '!']


Removing or keeping the whitespaces depends upon our application. For example, if we are making an LLM to write code then we must keep the whitespaces as they are very significant in coding.

Now we will remove all kind of non-alphanumeric characters from our text.

In [165]:
preprocessed = re.split(r'([, - " : ; . ? (){}!\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['CHAPTER', 'I', '"', 'Well', ',', 'Prince', ',', 'so', 'Genoa', 'and', 'Lucca', 'are', 'now', 'just', 'family', 'estates', 'of', 'the', 'Buonapartes', '.', 'But', 'I', 'warn', 'you', ',', 'if', 'you', 'don', "'", 't']


## Step 2 - Convert Tokens into Token IDs

In [166]:
all_words = sorted(set(preprocessed))
all_words = [w.strip() for w in all_words]
vocab_size = len(all_words)

print(vocab_size)

19709


In [167]:
vocab = {token:integer for integer, token in enumerate(all_words)} #takes each token and assigns an integer to it

In [168]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
('*', 5)
('*Catherine', 6)
('*Do', 7)
('*Kutuzov', 8)
('*Poisonous', 9)
(',', 10)


Now we also need a reverse mapping that can convert the token ids back into text. So we will create a complete tokenizer class which has both the methods.

**_Encode Method:_**
Raw Test ---> Tokenized Test ---> Token IDs

**_Decode Method:_**
Token IDs ---> Tokenized Text ---> Raw Text

In [169]:
class SimpleTokenizerV1:
    """
    A simple regex based tokenizer that converts text into token IDs
    and decodes token IDs back to text.
    """
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in self.str_to_int.items()}

    def encode(self, text):

        preprocessed = re.split(r'([,.:;?_!()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):

        text = " ".join([self.int_to_str[i] for i in ids]) #converting ids into individual tokens then join them
        text = re.sub(r'([,.:;?_!()\']|--|\s)', r'\1', text) #replace spaces before the specified punctuations
        return text

## Usage Example
---

In [170]:
tokenizer1 = SimpleTokenizerV1(vocab)

In [171]:
sample_text ="""
It was in July, 1805, and the speaker was the well-known Anna Pavlovna
Scherer, maid of honor and favorite of the Empress Marya Fedorovna. With
these words she greeted Prince Vasili Kuragin, a man of high rank and
importance, who was the first to arrive at her reception. Anna Pavlovna
had had a cough for some days. She was, as she said, suffering from la
grippe; grippe being then a new word in St. Petersburg, used only by the
elite.
"""

ids = tokenizer1.encode(sample_text)
print(ids)

[1458, 19188, 10666, 1502, 10, 57, 10, 3823, 17777, 16790, 19188, 17777, 19297, 229, 2185, 2527, 10, 11976, 13029, 10303, 3823, 8787, 13029, 17777, 924, 1811, 1014, 31, 3190, 17812, 19545, 16177, 9810, 2310, 3051, 1598, 10, 3292, 12031, 13029, 10205, 14662, 3823, 10618, 10, 19395, 19188, 17777, 8949, 17995, 4063, 4179, 10171, 14803, 31, 229, 2185, 9944, 9944, 3292, 6477, 9144, 16695, 6838, 31, 2612, 19188, 10, 4091, 16177, 15659, 10, 17365, 9355, 11438, 9830, 116, 9830, 4568, 17794, 3292, 12778, 19542, 10666, 2720, 31, 2208, 10, 18810, 13094, 5200, 17777, 8029, 31]


Now we will try to convert these token ids back into tokens.

In [172]:
tokens = tokenizer1.decode(ids)
print(tokens)

It was in July , 1805 , and the speaker was the well-known Anna Pavlovna Scherer , maid of honor and favorite of the Empress Marya Fedorovna . With these words she greeted Prince Vasili Kuragin , a man of high rank and importance , who was the first to arrive at her reception . Anna Pavlovna had had a cough for some days . She was , as she said , suffering from la grippe ; grippe being then a new word in St . Petersburg , used only by the elite .


Problem - There is an issue with this tokenizer class. If the encode function encounters any word that is not present in the vocab then it can not encode it and it will throw an error. That is why LLMs are trained with a large vocabulary.

### Adding Special Context Tokens
We will modify this tokenizer to handle unknown tokens. We will add two more tokens in the vocab at the end - <|unk|>, <|endoftext|>.
* <|unk|> is assigned to an unknown token in the vocab.
* While working with multiple text sources we add <|endoftext|> in between these sources. This was used in GPT aswell as this allows the LLM to understand the text better.

In [173]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [174]:
class SimpleTokenizerV2:
    """
    A simple regex based tokenizer that converts text into token IDs
    and decodes token IDs back to text.
    """
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in self.str_to_int.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
            ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text



## Usage Example
---

In [175]:
from src.tokenizer import SimpleTokenizer
tokenizer2 = SimpleTokenizer(vocab)

text1 = """Well, Prince, so Genoa and Lucca are now just family estates of the
Buonapartes. But I warn you, if you don't tell me that this means war,
if you still try to defend the infamies and horrors perpetrated by that
Antichrist--I really believe he is Antichrist"""

text2 = """
All her invitations without exception, written in French, and delivered hello
by a scarlet-liveried footman that morning, ran as follows:
"""

text = "<|endoftext|>".join((text1, text2))

print(text)

Well, Prince, so Genoa and Lucca are now just family estates of the
Buonapartes. But I warn you, if you don't tell me that this means war,
if you still try to defend the infamies and horrors perpetrated by that
Antichrist--I really believe he is Antichrist<|endoftext|>
All her invitations without exception, written in French, and delivered hello
by a scarlet-liveried footman that morning, ran as follows:



In [176]:
tokenizer2.encode(text)

[3149,
 10,
 2310,
 10,
 16633,
 1155,
 3823,
 1726,
 4016,
 12908,
 11322,
 8713,
 8338,
 13029,
 17777,
 507,
 31,
 514,
 1382,
 19177,
 19674,
 10,
 10494,
 19674,
 7667,
 2,
 17601,
 17696,
 12173,
 17772,
 17853,
 12188,
 19160,
 10,
 10494,
 19674,
 17081,
 18296,
 17995,
 6967,
 17777,
 10832,
 3823,
 10340,
 13627,
 5200,
 17772,
 235,
 12,
 1382,
 14745,
 4577,
 10096,
 11208,
 19710,
 180,
 10171,
 11155,
 19488,
 8452,
 10,
 19628,
 10666,
 1106,
 10,
 3823,
 7032,
 19710,
 5200,
 3292,
 15776,
 9138,
 17772,
 12512,
 10,
 14655,
 4091,
 9119,
 115]

In [177]:
tokenizer2.decode(tokenizer2.encode(text))

"Well, Prince, so Genoa and Lucca are now just family estates of the Buonapartes. But I warn you, if you don' t tell me that this means war, if you still try to defend the infamies and horrors perpetrated by that Antichrist -- I really believe he is <|unk|> All her invitations without exception, written in French, and delivered <|unk|> by a scarlet-liveried footman that morning, ran as follows:"

## Conclusion
---
Now we are able to handle unknown words effectively.

**Limitations of this tokenizer**:
* Not subword based
* Vocabulary grows large
* Cannot handle rare words efficiently
* Modern LLMs use BPE / WordPiece / SentencePiece

Some researchers also consider additional special tokens such
as the following:

**_[BOS] (beginning of sequence):_** This token marks the start of a text. It
signifies to the LLM where a piece of content begins.

**_[EOS] (end of sequence):_** This token is positioned at the end of a text,
and is especially useful when concatenating multiple unrelated texts,
similar to <|endoftext|>. For instance, when combining two different
Wikipedia articles or books, the [EOS] token indicates where one article
ends and the next one begins.

**_[PAD] (padding):_** When training LLMs with batch sizes larger than one,
the batch might contain texts of varying lengths. To ensure all texts have
the same length, the shorter texts are extended or "padded" using the
[PAD] token, up to the length of the longest text in the batch.


The tokenizer used for GPT models does not need any of these tokens mentioned
above but only uses an <|endoftext|> token for simplicity

The tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks
down words into subword units
